# Module 3 — Class 5: Data Pipelines (and a note on imbalance)

**Lecture reference:** `MODULE_3_OLIST_REWRITE.md` § Class 3-5

**Today:** turn your messy notebook code into a reproducible `sklearn.Pipeline`. Plus a quick look at the imbalanced `is_late` target — only 7% positive.

## Why this matters today

So far cleaning lives in scattered cells. When M4 starts you'll never re-run that mess. And in production, when **new** orders arrive, you need to apply the *exact same* transformations. A pipeline guarantees that.

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

df = pd.read_parquet('orders_step3.parquet').dropna(subset=['is_late'])
print(df.shape, '   is_late rate:', df['is_late'].mean().round(3))

## Section 1 — Define the column groups

In [ ]:
NUM = ['distance_km', 'log_freight', 'estimate_days',
       'purchase_hour', 'num_items', 'total_price']
CAT = ['payment_type']
TARGET = 'is_late'

X = df[NUM + CAT]
y = df[TARGET]

## Section 2 — Build the ColumnTransformer

In [ ]:
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, NUM),
    ('cat', categorical_pipe, CAT),
])

## Section 3 — Wrap with a model and demonstrate end-to-end

Note `class_weight='balanced'` — handles the 7% imbalance.

In [ ]:
full_pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(class_weight='balanced',
                               max_iter=1000, random_state=42)),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

full_pipe.fit(X_train, y_train)
y_pred = full_pipe.predict(X_test)
print(classification_report(y_test, y_pred))

## Section 4 — Save the pipeline (for M4 onwards)

**One file = production-ready preprocessing.**

In [ ]:
import joblib
joblib.dump(preprocessor, 'olist_preprocessor.pkl')
print('Saved olist_preprocessor.pkl')

## Section 5 — A note on imbalance handling

We used `class_weight='balanced'` here. Other tools you'll meet:

| Tool | When |
|---|---|
| `class_weight='balanced'` | Default — works for most models |
| `imbalanced-learn SMOTE` | Synthetic oversampling — when you have very few positives |
| Threshold tuning | Lower the 0.5 cutoff — covered in M4-4 |
| Cost-sensitive metrics (PR-AUC, F1) | Use these instead of accuracy |

## ✅ Quick Check

1. Why fit the scaler on training data only, not the full dataset?  
2. What does `OneHotEncoder(handle_unknown='ignore')` do, and why is it important for production?  
3. With `is_late` at 7%, what accuracy can a 'predict always 0' model achieve? Is it useful?

## 📝 Homework (Bronze)

Run the full pipeline. Submit `olist_preprocessor.pkl`.

## 🔥 Homework (Gold)

1. Try SMOTE oversampling (install `imbalanced-learn`). Compare F1 vs `class_weight='balanced'`.  
2. Half-page rationale: which imbalance handling would you choose for Olist and why?